In [1]:
import sqlite3
import json
from pathlib import Path
from datetime import datetime

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
db_path = root / "data" / "novamind.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS campaigns (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    created_at TEXT,
    topic TEXT,
    blog_title TEXT,
    blog_outline TEXT,
    blog_draft TEXT,
    founder_newsletter TEXT,
    ops_newsletter TEXT,
    creative_newsletter TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS performance_metrics (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    campaign_id INTEGER,
    persona TEXT,
    send_date TEXT,
    open_rate REAL,
    click_rate REAL,
    unsubscribe_rate REAL,
    subject_line_style TEXT,
    content_angle TEXT,
    cta_type TEXT,
    weighted_score REAL,
    FOREIGN KEY (campaign_id) REFERENCES campaigns(id)
)
""")

conn.commit()
print("Database ready at:", db_path)

Database ready at: /Users/tahadoguhancamkerten/Desktop/novamind-ai-pipeline/data/novamind.db


In [2]:
import json
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
exports_dir = root / "data" / "exports"

latest_file = sorted(exports_dir.glob("campaign_*.json"))[-1]

with open(latest_file, "r") as f:
    data = json.load(f)

topic = "AI in creative automation"

cursor.execute("""
INSERT INTO campaigns (
    created_at, topic, blog_title, blog_outline, blog_draft,
    founder_newsletter, ops_newsletter, creative_newsletter
) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", (
    datetime.now().isoformat(),
    topic,
    data["blog_title"],
    json.dumps(data["blog_outline"]),
    data["blog_draft"],
    data["newsletters"].get("Agency Founder", ""),
    data["newsletters"].get("Operations Manager", ""),
    data["newsletters"].get("Creative Lead", "")
))

conn.commit()
campaign_id = cursor.lastrowid
print("Inserted campaign:", campaign_id)

Inserted campaign: 15


In [3]:
sample_metrics = [
    {
        "persona": "Agency Founder",
        "open_rate": 0.41,
        "click_rate": 0.12,
        "unsubscribe_rate": 0.010,
        "subject_line_style": "growth-focused",
        "content_angle": "scaling without headcount",
        "cta_type": "book demo"
    },
    {
        "persona": "Operations Manager",
        "open_rate": 0.46,
        "click_rate": 0.16,
        "unsubscribe_rate": 0.005,
        "subject_line_style": "process-focused",
        "content_angle": "workflow efficiency",
        "cta_type": "see workflow example"
    },
    {
        "persona": "Creative Lead",
        "open_rate": 0.38,
        "click_rate": 0.09,
        "unsubscribe_rate": 0.015,
        "subject_line_style": "creative-time-focused",
        "content_angle": "protecting creative time",
        "cta_type": "read blog"
    },
    {
        "persona": "Account / Client Services Lead",
        "open_rate": 0.43,
        "click_rate": 0.13,
        "unsubscribe_rate": 0.007,
        "subject_line_style": "client-service-focused",
        "content_angle": "client visibility and responsiveness",
        "cta_type": "see delivery example"
    },
    {
        "persona": "Strategy / Marketing Lead",
        "open_rate": 0.40,
        "click_rate": 0.11,
        "unsubscribe_rate": 0.009,
        "subject_line_style": "planning-focused",
        "content_angle": "strategic leverage and execution speed",
        "cta_type": "read use case"
    }
]

for row in sample_metrics:
    cursor.execute("""
    INSERT INTO performance_metrics (
        campaign_id, persona, send_date, open_rate, click_rate, unsubscribe_rate,
        subject_line_style, content_angle, cta_type, weighted_score
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        campaign_id,
        row["persona"],
        datetime.now().isoformat(),
        row["open_rate"],
        row["click_rate"],
        row["unsubscribe_rate"],
        row["subject_line_style"],
        row["content_angle"],
        row["cta_type"],
        None
    ))

conn.commit()
print("Inserted", len(sample_metrics), "performance rows")

Inserted 5 performance rows


In [4]:
cursor.execute("""
SELECT id, open_rate, click_rate, unsubscribe_rate
FROM performance_metrics
WHERE campaign_id = ?
""", (campaign_id,))

rows = cursor.fetchall()

for row_id, open_rate, click_rate, unsubscribe_rate in rows:
    weighted_score = round(
        0.3 * open_rate +
        0.6 * click_rate -
        0.1 * unsubscribe_rate,
        4
    )

    cursor.execute("""
    UPDATE performance_metrics
    SET weighted_score = ?
    WHERE id = ?
    """, (weighted_score, row_id))

conn.commit()
print("Weighted scores saved")

Weighted scores saved


In [5]:
cursor.execute("""
SELECT persona, open_rate, click_rate, unsubscribe_rate, weighted_score
FROM performance_metrics
WHERE campaign_id = ?
ORDER BY weighted_score DESC
""", (campaign_id,))

rows = cursor.fetchall()

for row in rows:
    print(row)

('Operations Manager', 0.46, 0.16, 0.005, 0.2335)
('Account / Client Services Lead', 0.43, 0.13, 0.007, 0.2063)
('Agency Founder', 0.41, 0.12, 0.01, 0.194)
('Strategy / Marketing Lead', 0.4, 0.11, 0.009, 0.1851)
('Creative Lead', 0.38, 0.09, 0.015, 0.1665)
